In [ ]:
# =============================================================================
# KAN - ECG CLASSIFICATION + TINYML EXPORT (ESP32)
# Versão unificada e corrigida
# =============================================================================

import os
import re
import random
import numpy as np
import pandas as pd
import sympy as sp
import joblib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import pywt

from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

from kan import KAN


# =============================================================================
# BLOCO 1 — EXTRAÇÃO DE FEATURES (ÚNICA VERSÃO, usada em treino e geração do .h)
# =============================================================================

def extract_wavelet_features(signal, wavelet='db4', level=4):
    """
    Retorna a energia de cada banda wavelet (aproximação + detalhes).
    level=4 → 5 coeficientes no total.
    """
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    return [float(np.sum(c ** 2)) for c in coeffs]


def spectral_band_features(signal, fs=360):
    """
    Energia espectral em três bandas de frequência:
      band1: 0–5 Hz   (ondas lentas, linha de base)
      band2: 5–15 Hz  (complexo QRS principal)
      band3: 15–40 Hz (atividade de alta frequência, ruído muscular)
    """
    yf = np.abs(np.fft.rfft(signal))
    xf = np.fft.rfftfreq(len(signal), 1 / fs)

    band1 = float(yf[(xf >= 0)  & (xf < 5)].sum())
    band2 = float(yf[(xf >= 5)  & (xf < 15)].sum())
    band3 = float(yf[(xf >= 15) & (xf < 40)].sum())

    return [band1, band2, band3]


def extract_features(signal, fs=360):
    """
    Extrai todas as features usadas no treino.
    Esta é a ÚNICA função de extração — usada tanto para montar o dataset
    de treino/validação quanto para gerar o dataset.h para o ESP32.

    Ordem das features (total = 22):
      [0-10]  Estatísticas temporais (11 features)
      [11]    Amplitude do pico R
      [12]    Largura do QRS (segundos)
      [13]    Frequência dominante (Hz)
      [14]    Energia espectral total (FFT)
      [15-17] Bandas espectrais (3 features)
      [18-22] Energias wavelet db4 level=4 (5 features)
    """
    signal = signal.astype(np.float32)

    # ---- Estatísticas temporais ----
    features = [
        float(np.mean(signal)),
        float(np.std(signal)),
        float(np.min(signal)),
        float(np.max(signal)),
        float(np.ptp(signal)),                         # peak-to-peak
        float(np.sum(signal ** 2)),                    # energia total
        float(np.sum(np.abs(signal))),                 # soma dos valores absolutos
        float(np.max(np.abs(np.diff(signal)))),        # maior variação entre amostras
        float(len(signal)),                            # comprimento do sinal
        float(skew(signal)),
        float(kurtosis(signal)),
    ]

    # ---- Detecção do pico R ----
    peaks, _ = find_peaks(signal, distance=int(0.4 * fs))

    if len(peaks) > 0:
        # Seleciona o pico mais próximo do centro do segmento
        center = len(signal) // 2
        peak = peaks[np.argmin(np.abs(peaks - center))]
        r_peak_amp = float(signal[peak])
    else:
        peak = -1
        r_peak_amp = 0.0

    features.append(r_peak_amp)

    # ---- Largura do QRS ----
    if peak != -1:
        threshold = signal[peak] * 0.5

        left = peak
        while left > 0 and signal[left] > threshold:
            left -= 1

        right = peak
        while right < len(signal) - 1 and signal[right] > threshold:
            right += 1

        qrs_width = float((right - left) / fs)
    else:
        qrs_width = 0.0

    features.append(qrs_width)

    # ---- FFT: frequência dominante e energia total ----
    yf = np.abs(rfft(signal))
    xf = rfftfreq(len(signal), 1 / fs)

    dominant_freq   = float(xf[np.argmax(yf)])
    spectral_energy = float(np.sum(yf))

    features.append(dominant_freq)
    features.append(spectral_energy)

    # ---- Bandas espectrais ----
    features.extend(spectral_band_features(signal, fs))

    # ---- Wavelet ----
    features.extend(extract_wavelet_features(signal, wavelet='db4', level=4))

    return features   # 22 features no total


# =============================================================================
# BLOCO 2 — CONSTRUÇÃO DO DATASET
# =============================================================================

def build_dataset(root_dir, fs=360):
    """
    Percorre root_dir/<classe>/*.csv e extrai features de cada sinal.
    Detecta automaticamente a coluna correta no CSV.
    Retorna X (features), y (nomes das classes).
    """
    X, y = [], []

    for class_name in sorted(os.listdir(root_dir)):
        class_path = os.path.join(root_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            if not file.endswith(".csv"):
                continue

            df = pd.read_csv(os.path.join(class_path, file))

            if "amplitude" in df.columns:
                signal = df["amplitude"].values
            elif "channel_0" in df.columns:
                signal = df["channel_0"].values
            else:
                signal = df.values.flatten()

            X.append(extract_features(signal, fs))
            y.append(class_name)

    return np.array(X, dtype=np.float32), np.array(y)


# =============================================================================
# BLOCO 3 — CARREGAMENTO DOS DADOS E PRÉ-PROCESSAMENTO
# =============================================================================

print("=" * 60)
print("Carregando datasets...")
print("=" * 60)

X_train, y_train = build_dataset("dataset_split_2/train")
X_val,   y_val   = build_dataset("dataset_split_2/val")

print(f"Treino:    {X_train.shape[0]} amostras | {X_train.shape[1]} features")
print(f"Validação: {X_val.shape[0]} amostras  | {X_val.shape[1]} features")

# Codificação dos rótulos
encoder      = LabelEncoder()
y_train_enc  = encoder.fit_transform(y_train)
y_val_enc    = encoder.transform(y_val)
num_classes  = len(encoder.classes_)
class_names  = list(encoder.classes_)

print(f"Classes ({num_classes}): {class_names}")

# Escalonamento — IMPORTANTE: fit apenas no treino
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

# Serializa o scaler para reutilização futura (ex.: generate_dataset_h)
os.makedirs("ArduinoCode", exist_ok=True)
joblib.dump(scaler,  "ArduinoCode/scaler.pkl")
joblib.dump(encoder, "ArduinoCode/encoder.pkl")
print("Scaler e encoder salvos em ArduinoCode/")

# Conversão para tensores PyTorch
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_scaled,   dtype=torch.float32)
y_train_t = torch.tensor(y_train_enc,    dtype=torch.long)
y_val_t   = torch.tensor(y_val_enc,      dtype=torch.long)


# =============================================================================
# BLOCO 4 — DEFINIÇÃO E TREINO DA KAN
# =============================================================================

n_features = X_train_t.shape[1]

model = KAN(
    width=[n_features, 16, 8, num_classes],
    grid=5,
    k=3,
)

dataset = {
    'train_input': X_train_t,
    'train_label': y_train_t,
    'test_input':  X_val_t,
    'test_label':  y_val_t,
}

# Pesos para lidar com desbalanceamento de classes
pesos_classes = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_enc),
    y=y_train_enc,
)
pesos_tensor = torch.tensor(pesos_classes, dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss(weight=pesos_tensor)

print("\nIniciando treinamento da KAN...")
results = model.fit(
    dataset,
    opt="Adam",
    steps=25000,
    lr=0.001,
    loss_fn=loss_fn,
    batch=512,
)


# =============================================================================
# BLOCO 5 — AVALIAÇÃO E MATRIZ DE CONFUSÃO
# =============================================================================

with torch.no_grad():
    y_pred_logits = model(X_val_t)
    y_pred_idx    = torch.argmax(y_pred_logits, dim=1).numpy()

y_pred_labels = encoder.inverse_transform(y_pred_idx)

cm = confusion_matrix(y_val, y_pred_labels, labels=class_names)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues",
)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão — KAN (ECG)")
plt.tight_layout()
plt.savefig("ArduinoCode/confusion_matrix.png", dpi=150)
plt.show()

print("\n" + classification_report(y_val, y_pred_labels, digits=4))




In [ ]:
# =============================================================================
# BLOCO 6 — CONVERSÃO PARA C++ / TINYML
# =============================================================================

print("\n" + "=" * 60)
print("Iniciando conversão para TinyML (C++)...")
print("=" * 60)

# Poda: remove conexões com peso negligenciável
model_pruned = model.prune()

# Regressão simbólica: converte splines em fórmulas matemáticas fechadas
# Bibliotecas compatíveis com std:: do C++
model_pruned.auto_symbolic(
    lib=['x', 'x^2', 'x^3', 'x^4', 'exp', 'log', 'sqrt', 'tanh', 'sin', 'abs']
)

with torch.no_grad():
    formulas_list = model_pruned.symbolic_formula()[0]

# Número de entradas APÓS a poda (pode ser menor que n_features)
input_size_pruned = model_pruned.width[0][0] if isinstance(model_pruned.width[0], list) else model_pruned.width[0]
print(f"Entradas após poda: {input_size_pruned} (original: {n_features})")


def transform_expression(equation):
    """
    Converte uma expressão SymPy/string para C++:
      - substitui ** por ^ (para o parser do SymPy)
      - converte potências (Pow) para a função pow()
    """
    modified_eq = str(equation).replace('**', '^')
    expr = sp.sympify(modified_eq)

    def replace_pow(e):
        if e.is_Pow:
            base, exp = e.args
            return sp.Function('pow')(replace_pow(base), replace_pow(exp))
        elif e.is_Function:
            return e.func(*[replace_pow(a) for a in e.args])
        elif e.is_Add or e.is_Mul:
            return e.func(*[replace_pow(a) for a in e.args])
        else:
            return e

    return str(replace_pow(expr))


def safe_replace_math(formula_string):
    """
    Substitui nomes de funções matemáticas pelos equivalentes std:: do C++
    usando regex com word-boundary (\b) para evitar colisões como
    'tanh' → 'std::tanh' e depois 'std::tanh' → 'std::std::tanh'.
    A ordem importa: funções mais longas primeiro (tanh antes de tan).
    """
    replacements = [
        (r'\btanh\b',    'std::tanh'),
        (r'\bsinh\b',    'std::sinh'),
        (r'\bcosh\b',    'std::cosh'),
        (r'\basin\b',    'std::asin'),    # arcsin já convertido
        (r'\bacos\b',    'std::acos'),
        (r'\batan\b',    'std::atan'),
        (r'\bsin\b',     'std::sin'),
        (r'\bcos\b',     'std::cos'),
        (r'\btan\b',     'std::tan'),
        (r'\bsqrt\b',    'std::sqrt'),
        (r'\bexp\b',     'std::exp'),
        (r'\blog\b',     'std::log'),
        (r'\babs\b',     'std::abs'),
        (r'\bpow\b',     'std::pow'),
        (r'\barcsin\b',  'std::asin'),
        (r'\barctan\b',  'std::atan'),
        (r'\barctanh\b', 'std::atanh'),
    ]
    for pattern, replacement in replacements:
        formula_string = re.sub(pattern, replacement, formula_string)
    return formula_string


def generate_kan_header(formulas, input_size, num_classes, class_names):
    """
    Gera o arquivo kan_model.h contendo:
      - Mapeamento de índice → nome da classe
      - Função predict() que recebe as features escalonadas e retorna
        o índice da classe com maior logit (ArgMax)
    """
    lines = [
        "#ifndef KAN_MODEL_H",
        "#define KAN_MODEL_H",
        "",
        "#include <cmath>",
        "#include <cfloat>",
        "",
        "// -------------------------------------------------------",
        "// Modelo KAN — ECG Classification",
        f"// Entradas : {input_size} features (StandardScaler aplicado)",
        f"// Saídas   : {num_classes} classes",
        "// -------------------------------------------------------",
        "",
        "// Mapeamento: índice → nome da classe",
        f"// {', '.join([f'{i}={c}' for i, c in enumerate(class_names)])}",
        "",
    ]

    # Assinatura da função
    params = ", ".join([f"float x_{i}" for i in range(1, input_size + 1)])
    lines.append(f"int predict({params}) {{")
    lines.append(f"    float logits[{num_classes}];")
    lines.append("")

    for i in range(num_classes):
        formula_str = transform_expression(formulas[i])
        formula_str = safe_replace_math(formula_str)

        lines.append(f"    // Classe {i}: {class_names[i]}")
        lines.append(f"    logits[{i}] = {formula_str};")
        lines.append("")

    # ArgMax
    lines += [
        "    // ArgMax — retorna o índice da classe com maior logit",
        "    int   best_class = 0;",
        "    float max_logit  = logits[0];",
        f"    for (int i = 1; i < {num_classes}; i++) {{",
        "        if (logits[i] > max_logit) {",
        "            max_logit  = logits[i];",
        "            best_class = i;",
        "        }",
        "    }",
        "    return best_class;",
        "}",
        "",
        "#endif // KAN_MODEL_H",
    ]

    return "\n".join(lines)


cpp_code = generate_kan_header(
    formulas_list,
    input_size=input_size_pruned,
    num_classes=num_classes,
    class_names=class_names,
)

with open("ArduinoCode/kan_model.h", "w") as f:
    f.write(cpp_code)

print("kan_model.h salvo em ArduinoCode/")


# =============================================================================
# BLOCO 7 — GERAÇÃO DO dataset.h PARA VALIDAÇÃO NO ESP32
# =============================================================================

def generate_dataset_h(
    dataset_path,
    scaler,
    encoder,
    samples_per_class=10,
    output="ArduinoCode/dataset.h",
    fs=360,
):
    """
    Gera dataset.h com amostras escalonadas prontas para uso no ESP32.
    Usa a MESMA função extract_features() do treino e o MESMO scaler,
    garantindo consistência total entre Python e o dispositivo embarcado.
    """
    X, y = [], []

    classes = sorted([
        c for c in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, c))
    ])

    for label_idx, class_name in enumerate(classes):
        class_path = os.path.join(dataset_path, class_name)
        files = [f for f in os.listdir(class_path) if f.endswith(".csv")]
        random.shuffle(files)

        loaded = 0
        for file in files:
            if loaded >= samples_per_class:
                break

            df = pd.read_csv(os.path.join(class_path, file))

            if "channel_0" in df.columns:
                signal = df["channel_0"].values
            elif "amplitude" in df.columns:
                signal = df["amplitude"].values
            else:
                signal = df.values.flatten()

            X.append(extract_features(signal, fs))
            y.append(label_idx)
            loaded += 1

    X = np.array(X, dtype=np.float32)
    y = np.array(y)

    # Escalonamento com o MESMO scaler do treino
    X_scaled = scaler.transform(X)

    with open(output, "w") as f:
        f.write(f"// Auto-gerado por kolmo_corrigido.py\n")
        f.write(f"// Classes: {', '.join([f'{i}={c}' for i, c in enumerate(classes)])}\n\n")
        f.write(f"#define NUM_SAMPLES  {len(X_scaled)}\n")
        f.write(f"#define NUM_FEATURES {X_scaled.shape[1]}\n")
        f.write(f"#define NUM_CLASSES  {len(classes)}\n\n")

        f.write("float dataset[NUM_SAMPLES][NUM_FEATURES] = {\n")
        for row in X_scaled:
            values = ", ".join([f"{v:.6f}" for v in row])
            f.write(f"    {{ {values} }},\n")
        f.write("};\n\n")

        f.write("int labels[NUM_SAMPLES] = {\n    ")
        f.write(", ".join(map(str, y)))
        f.write("\n};\n\n")

        f.write('const char* class_names[] = {\n')
        for c in classes:
            f.write(f'    "{c}",\n')
        f.write("};\n")

    print(f"dataset.h gerado: {len(X_scaled)} amostras × {X_scaled.shape[1]} features → {output}")


generate_dataset_h(
    dataset_path="dataset_split_2/val",
    scaler=scaler,
    encoder=encoder,
    samples_per_class=100,
)

print("\n" + "=" * 60)
print("Arquivos gerados em ArduinoCode/:")
print("  kan_model.h       — modelo C++ para ESP32")
print("  dataset.h         — amostras de validação escalonadas")
print("  scaler.pkl        — StandardScaler serializado")
print("  encoder.pkl       — LabelEncoder serializado")
print("  confusion_matrix.png")
print("=" * 60)


In [3]:
# =============================================================================
# gerar_scaler_h.py
# Execute este script UMA VEZ após o treino para gerar o scaler.h correto.
# Ele lê o scaler.pkl salvo pelo kolmo_corrigido.py e exporta os valores
# de média e desvio para C++.
# =============================================================================

import joblib
import numpy as np

scaler  = joblib.load("ArduinoCode/scaler.pkl")
encoder = joblib.load("ArduinoCode/encoder.pkl")

means       = scaler.mean_
stds        = scaler.scale_
num_features = len(means)
num_classes  = len(encoder.classes_)
class_names  = list(encoder.classes_)

feature_names = [
    "mean(signal)",
    "std(signal)",
    "min(signal)",
    "max(signal)",
    "ptp(signal)",
    "energy total",
    "sum_abs",
    "max_diff",
    "len(signal)",
    "skewness",
    "kurtosis",
    "r_peak_amp",
    "qrs_width",
    "dominant_freq",
    "spectral_energy",
    "band1 (0-5 Hz)",
    "band2 (5-15 Hz)",
    "band3 (15-40 Hz)",
    "wavelet energy lv1",
    "wavelet energy lv2",
    "wavelet energy lv3",
    "wavelet energy lv4",
    "wavelet energy approx",
]

lines = []
lines.append("// scaler.h — gerado automaticamente por gerar_scaler_h.py")
lines.append("// NÃO edite manualmente.")
lines.append("")
lines.append("#ifndef SCALER_H")
lines.append("#define SCALER_H")
lines.append("")
lines.append(f"#define NUM_FEATURES {num_features}")
lines.append(f"#define NUM_CLASSES  {num_classes}")
lines.append("")

# Nomes das classes como comentário
class_map = ", ".join([f"{i}={c}" for i, c in enumerate(class_names)])
lines.append(f"// Classes: {class_map}")
lines.append("")

lines.append("const float SCALER_MEAN[] = {")
for i, (v, name) in enumerate(zip(means, feature_names)):
    lines.append(f"    {v:.8f}f,  // feat {i:2d} — {name}")
lines.append("};")
lines.append("")

lines.append("const float SCALER_STD[] = {")
for i, (v, name) in enumerate(zip(stds, feature_names)):
    lines.append(f"    {v:.8f}f,  // feat {i:2d} — {name}")
lines.append("};")
lines.append("")
lines.append("#endif // SCALER_H")

output = "ArduinoCode/scaler.h"
with open(output, "w") as f:
    f.write("\n".join(lines))

print(f"scaler.h gerado em {output}")
print(f"  Features : {num_features}")
print(f"  Classes  : {num_classes} → {class_names}")


scaler.h gerado em ArduinoCode/scaler.h
  Features : 23
  Classes  : 8 → ['A', 'B', 'F', 'L', 'N', 'R', 'V', 'f']
